In [12]:
import pandas as pd

path = "armada.dat"

#fixed-width file: specifying colspecs to correctly parse columns
colspecs = [
    (0, 28),   #battle name (cols 1–28)
    (30, 34),  #year (31–34)
    (40, 42),  #portuguese ships (41–42)
    (48, 50),  #dutch ships (49–50)
    (56, 58),  #english ships (57–58)
    (61, 66),  #ratio of portuguese to dutch/british ships (62–66)
    (73, 74),  #spanish involvement (74)  1=yes, 0=no
    (80, 82),  #portuguese outcome (81–82) -1=defeat, 0=draw, 1=victory
]

names = [
    "battle",
    "year",
    "port_ships",
    "dutch_ships",
    "eng_ships",
    "ratio_port_to_enemy",
    "spanish_involvement",
    "port_outcome",
]

df = pd.read_fwf(path, colspecs=colspecs, names=names)
df

,battle,year,port_ships,dutch_ships,eng_ships,ratio_port_to_enemy,spanish_involvement,port_outcome
0,Bantam,1601,6,3,0,2.000,0,0
1,Malacca Strait,1606,14,11,0,1.273,0,0
2,Ilha das Naus,1606,6,9,0,0.667,0,-1
3,Pulo Butum,1606,7,9,0,0.778,0,1
4,Surrat,1615,6,0,4,1.500,0,0
5,Ilha das Naus,1615,3,5,0,0.600,0,-1
6,Jask,1620,4,0,4,1.000,0,0
7,Hormuz,1622,6,0,5,1.200,0,-1
8,Mogincoal Shoals,1622,4,4,2,0.667,0,-1
9,Hormuz,1625,8,4,4,1.000,0,0


In [2]:
df.isna().sum()

battle                 0
year                   0
port_ships             0
dutch_ships            0
eng_ships              0
ratio_port_to_enemy    0
spanish_involvement    0
port_outcome           0
dtype: int64

In [3]:
df.shape

(28, 8)

In [4]:
df["port_outcome"].value_counts()

port_outcome
 0    13
-1    10
 1     5
Name: count, dtype: int64

In [5]:
#features and target
X = df[["port_ships", "dutch_ships", "eng_ships", "spanish_involvement"]]
y = df["port_outcome"]

In [6]:
#data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
#performing grid search to chose kernel

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        "kernel": ["linear"],
        "C": [0.1, 1, 10, 100]
    },
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", 0.1, 0.01, 0.001]
    },
    {
        "kernel": ["poly"],
        "C": [0.1, 1, 10],
        "degree": [2, 3, 4],
        "gamma": ["scale", 0.1, 0.01]
    }
]

grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1 
)

grid.fit(X_train_scaled, y_train)

,estimator,SVC()
,param_grid,"[{'C': [0.1, 1, ...], 'kernel': ['linear']}, {'C': [0.1, 1, ...], 'gamma': ['scale', 0.1, ...], 'kernel': ['rbf']}, ...]"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,0.1


In [8]:
#grid search results
results = pd.DataFrame(grid.cv_results_)
results[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
6,"{'C': 0.1, 'gamma': 0.01, 'kernel': 'rbf'}",0.52381,0.134687,1
10,"{'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}",0.52381,0.134687,1
23,"{'C': 0.1, 'degree': 3, 'gamma': 'scale', 'ker...",0.47619,0.067344,3
26,"{'C': 0.1, 'degree': 4, 'gamma': 'scale', 'ker...",0.47619,0.067344,3
27,"{'C': 0.1, 'degree': 4, 'gamma': 0.1, 'kernel'...",0.47619,0.067344,3
28,"{'C': 0.1, 'degree': 4, 'gamma': 0.01, 'kernel...",0.47619,0.067344,3
29,"{'C': 1, 'degree': 2, 'gamma': 'scale', 'kerne...",0.47619,0.067344,3
30,"{'C': 1, 'degree': 2, 'gamma': 0.1, 'kernel': ...",0.47619,0.067344,3
31,"{'C': 1, 'degree': 2, 'gamma': 0.01, 'kernel':...",0.47619,0.067344,3
32,"{'C': 1, 'degree': 3, 'gamma': 'scale', 'kerne...",0.47619,0.067344,3


In [9]:
#fit final RBF SVM model
from sklearn.metrics import classification_report, confusion_matrix

final_model = SVC(kernel='rbf', C=1, gamma=0.01)
final_model.fit(X_train_scaled, y_train)

preds = final_model.predict(X_test_scaled)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

[[0 2 0]
 [0 3 0]
 [0 2 0]]
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         2
           0       0.43      1.00      0.60         3
           1       0.00      0.00      0.00         2

    accuracy                           0.43         7
   macro avg       0.14      0.33      0.20         7
weighted avg       0.18      0.43      0.26         7



/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stevegon/SynologyDrive/code/uw/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [10]:
#fit knn model
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)

knn_preds = knn.predict(X_test_scaled)
print("Confusion Matrix (kNN):")
print(confusion_matrix(y_test, knn_preds))
print("\nClassification Report (kNN):")
print(classification_report(y_test, knn_preds, zero_division=0))

Confusion Matrix (kNN):
[[2 0 0]
 [2 1 0]
 [1 1 0]]

Classification Report (kNN):
              precision    recall  f1-score   support

          -1       0.40      1.00      0.57         2
           0       0.50      0.33      0.40         3
           1       0.00      0.00      0.00         2

    accuracy                           0.43         7
   macro avg       0.30      0.44      0.32         7
weighted avg       0.33      0.43      0.33         7



In [11]:
#fit decision tree model
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

tree_preds = tree.predict(X_test)
print("Confusion Matrix (Decision Tree):")
print(confusion_matrix(y_test, tree_preds))
print("\nClassification Report (Decision Tree):")
print(classification_report(y_test, tree_preds, zero_division=0))

Confusion Matrix (Decision Tree):
[[1 1 0]
 [1 2 0]
 [0 2 0]]

Classification Report (Decision Tree):
              precision    recall  f1-score   support

          -1       0.50      0.50      0.50         2
           0       0.40      0.67      0.50         3
           1       0.00      0.00      0.00         2

    accuracy                           0.43         7
   macro avg       0.30      0.39      0.33         7
weighted avg       0.31      0.43      0.36         7



**Report and compare their results with those from SVM.**

I compared three classifiers: SVM (RBF kernel), kNN, and a Decision Tree. All of them had difficulty predicting the Portuguese outcome because the dataset is very small and the classes are imbalanced. The SVM had the best cross-validation accuracy when I performed the Grid Search with training data, but on the test set it predicted only the majority class (0 = draw), giving it an accuracy of about 0.43. The kNN model behaved almost the same as the SVM. With so few examples, especially for the −1 and +1 classes, kNN mostly predicted the majority class as well. The Decision Tree made slightly more varied predictions but still struggled to correctly classify the minority classes, and its accuracy was similar to the other models. Overall, SVM performed best during training, but in practice all three models produced similar and limited results due to the small, imbalanced dataset. 